In [1]:
# bow vs tfidf

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os

import dagshub

d:\mlops\mlops-mini-project\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri("https://dagshub.com/kush2501/mlops-mini-project.mlflow")
dagshub.init(repo_owner='kush2501', repo_name='mlops-mini-project', mlflow=True)

Accessing as kush2501

Initialized MLflow to track repo "kush2501/mlops-mini-project"

Repository kush2501/mlops-mini-project initialized!

In [3]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:
# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


In [5]:
# Normalize the text data
df = normalize_text(df)
df.head()

,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin n bed headache ughhhh waitin call
2,sadness,funeral ceremony gloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [6]:
# keep only sadness and happiness
df = df[df['sentiment'].isin(['happiness','sadness'])]

# encode labels
df['sentiment'] = df['sentiment'].map({
    'sadness':0,
    'happiness':1
})

In [7]:
df = df[df['content'].str.strip() != ""]
df['content'] = df['content'].astype(str)

# Set the experiment name.
mlflow.set_experiment("Bow VS Tfidf")

<Experiment: artifact_location='mlflow-artifacts:/2f9feeea8f524651a2dc0655d9fa231e', creation_time=1773822845407, experiment_id='5', last_update_time=1773822845407, lifecycle_stage='active', name='Bow VS Tfidf', tags={}, workspace='default'>

In [8]:
# Define feature extraction methods
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

# Define algorithms
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}


In [ ]:
# Start the parent run
with mlflow.start_run(run_name="All Experiments") as parent_run:

    # Loop through algorithms and feature extraction methods (Child Runs)
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():

            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:

                X = df['content']
                y = df['sentiment']

                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                
                # Step 3: Vectorization (AFTER split)
                X_train_vec = vectorizer.fit_transform(X_train)
                X_test_vec = vectorizer.transform(X_test)

                # Log preprocessing parameters
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)
                
                # Model training
                model = algorithm
                model.fit(X_train_vec, y_train)
                
                # Log model parameters
                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)
                
                # Model evaluation
                y_pred = model.predict(X_test_vec)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)


                # 🔹 Step 8: Log ORIGINAL DATA (Not vectorized ❗)
                train_df = pd.DataFrame({"text": X_train.reset_index(drop=True),"target": y_train.reset_index(drop=True)})
                test_df = pd.DataFrame({"text": X_test.reset_index(drop=True),"target": y_test.reset_index(drop=True)})

                mlflow.log_input(mlflow.data.from_pandas(train_df),context="training")
                mlflow.log_input(mlflow.data.from_pandas(test_df),context="validation")
                
                # Log model
                mlflow.sklearn.log_model(model, name="model")
                
                # Save and log the notebook
                mlflow.log_artifact("exp2_bow_vs_tfidf.ipynb")
                
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")

d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing

Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7903614457831325
Precision: 0.7810707456978967
Recall: 0.7986314760508308
F1 Score: 0.7897535041082648
🏃 View run LogisticRegression with BoW at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/f293c3f65c614dfdb50a888052fe7a66
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing

Algorithm: LogisticRegression, Feature Engineering: TF-IDF
Accuracy: 0.7879518072289157
Precision: 0.784390243902439
Recall: 0.7859237536656891
F1 Score: 0.78515625
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/3ef49537cdd64abba64808120ed57b56
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing

...

Algorithm: RandomForest, Feature Engineering: TF-IDF
Accuracy: 0.7720481927710844
Precision: 0.7634099616858238
Recall: 0.7790811339198436
F1 Score: 0.7711659409772618
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/da9c1aa6183345b6816b986f31f58985
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing

Algorithm: GradientBoosting, Feature Engineering: BoW
Accuracy: 0.7320481927710843
Precision: 0.781664656212304
Recall: 0.6334310850439883
F1 Score: 0.6997840172786177
🏃 View run GradientBoosting with BoW at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/9061ba82b1b044c8b5bc9fe4233d82e7
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5


d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
d:\mlops\mlops-mini-project\myenv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing

Algorithm: GradientBoosting, Feature Engineering: TF-IDF
Accuracy: 0.7272289156626506
Precision: 0.8067114093959732
Recall: 0.5874877810361682
F1 Score: 0.6798642533936652
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/f334bfb5cc70458d834785fd03d8df4f
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5
🏃 View run All Experiments at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5/runs/8177d1e12c72479d9e559f7204f0f442
🧪 View experiment at: https://dagshub.com/kush2501/mlops-mini-project.mlflow/#/experiments/5
